[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IshtVibhu/iv-Mesa-ABM-Tutorial/blob/main/NB_09_NetworkX.ipynb)

# Python for Computational Economics
## Notebook 09 — NetworkX: Trade Networks and Financial Contagion

**Prerequisites:** NB_01–NB_04

**What you will learn:**
- Building and analysing directed trade networks
- Centrality measures: which countries are the most connected?
- Community detection: trading blocs
- Financial contagion simulation: spreading bank failures
- Sugarscape trade network: who traded with whom?

In [ ]:
try:
    import networkx as nx
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError:
    !pip install networkx numpy pandas matplotlib --quiet
    import networkx as nx
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
print(f"NetworkX {nx.__version__}")

---
## 1. Building a Trade Network

A **directed graph** `G` where each node is a country and each edge `(i, j)` represents exports from country `i` to country `j`, weighted by trade volume.

In [ ]:
# Stylised bilateral trade data (exports in billion USD, 2023 approximate)
trade_flows = [
    ("USA",     "China",   153),
    ("China",   "USA",     500),
    ("Germany", "USA",     135),
    ("USA",     "Germany",  67),
    ("China",   "Germany",  60),
    ("Germany", "France",   95),
    ("France",  "Germany",  78),
    ("Germany", "Italy",    65),
    ("Italy",   "Germany",  57),
    ("Japan",   "USA",     134),
    ("USA",     "Japan",    75),
    ("China",   "Japan",    84),
    ("Japan",   "China",   110),
    ("India",   "USA",      83),
    ("USA",     "India",    40),
    ("UK",      "USA",      58),
    ("USA",     "UK",      120),
    ("China",   "India",    85),
    ("India",   "China",    15),
    ("France",  "UK",       34),
    ("UK",      "France",   29),
]

G = nx.DiGraph()
for source, target, weight in trade_flows:
    G.add_edge(source, target, weight=weight)

print(f"Nodes (countries): {G.number_of_nodes()}")
print(f"Edges (trade flows): {G.number_of_edges()}")
print(f"Countries: {sorted(G.nodes())}")

# Total exports per country
total_exports = {n: sum(d["weight"] for _, _, d in G.out_edges(n, data=True)) for n in G.nodes()}
print("\nTotal exports ($bn):")
for country, exports in sorted(total_exports.items(), key=lambda x: -x[1]):
    print(f"  {country:<10} {exports:>5}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

pos = nx.spring_layout(G, seed=42, k=2)
weights = [G[u][v]["weight"] for u, v in G.edges()]
max_w   = max(weights)

nx.draw_networkx_nodes(G, pos, node_size=800, node_color="steelblue", alpha=0.8, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=9, font_color="white", ax=ax)
nx.draw_networkx_edges(
    G, pos,
    width=[w / max_w * 5 for w in weights],
    alpha=0.6,
    edge_color="gray",
    arrows=True,
    arrowsize=15,
    ax=ax
)

ax.set_title("Stylised Global Trade Network (2023)", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.savefig("trade_network.png", dpi=100, bbox_inches="tight")
plt.show()

---
## 2. Centrality — Which Country is Most Connected?

In [ ]:
# PageRank (used by Google, but also measures trade importance)
pagerank = nx.pagerank(G, weight="weight")

# In-degree centrality (sum of imports weighted)
in_strength = {n: sum(d["weight"] for _, _, d in G.in_edges(n, data=True)) for n in G.nodes()}

centrality_df = pd.DataFrame({
    "PageRank":   pagerank,
    "In_Strength": in_strength,
    "Out_Strength": total_exports,
}).sort_values("PageRank", ascending=False).round(3)

print("Trade Network Centrality:")
print(centrality_df)

# Betweenness centrality: how often a node lies on the shortest path between others
betweenness = nx.betweenness_centrality(G, weight="weight", normalized=True)
print("\nBetweenness centrality (trade intermediaries):")
for country, bc in sorted(betweenness.items(), key=lambda x: -x[1]):
    print(f"  {country:<10} {bc:.4f}")

---
## 3. Financial Contagion — Spreading Bank Failures

Banks are connected by interbank lending. If one bank fails, it may not repay its creditors, causing further failures — a cascade.

In [ ]:
np.random.seed(7)
N_BANKS  = 15
N_LINKS  = 30

# Create a random interbank network
bank_network = nx.erdos_renyi_graph(N_BANKS, p=0.25, directed=True, seed=7)

# Assign random exposures (loan amounts) and capital buffers
for u, v in bank_network.edges():
    bank_network[u][v]["exposure"] = np.random.uniform(0.5, 3.0)

capital = {n: np.random.uniform(1.0, 4.0) for n in bank_network.nodes()}

def simulate_contagion(G, capital, initial_failure, loss_given_default=0.5):
    """Propagate bank failures from an initial shock."""
    capital = capital.copy()
    failed  = {initial_failure}
    new_failures = {initial_failure}
    rounds  = 0

    while new_failures:
        rounds += 1
        next_failures = set()
        for failed_bank in new_failures:
            # Creditors of the failed bank lose exposure * LGD
            for creditor in G.predecessors(failed_bank):
                if creditor in failed:
                    continue
                loss = G[creditor][failed_bank]["exposure"] * loss_given_default
                capital[creditor] -= loss
                if capital[creditor] <= 0 and creditor not in failed:
                    next_failures.add(creditor)
        failed |= next_failures
        new_failures = next_failures

    return failed, rounds

# Simulate failure of the most-connected bank
most_connected = max(bank_network.nodes(), key=lambda n: bank_network.in_degree(n) + bank_network.out_degree(n))
failed_banks, rounds = simulate_contagion(bank_network, capital, initial_failure=most_connected)

print(f"Initial failure: Bank {most_connected}")
print(f"Total failures:  {len(failed_banks)} / {N_BANKS} banks ({len(failed_banks)/N_BANKS:.0%})")
print(f"Contagion rounds: {rounds}")
print(f"Surviving banks: {sorted(set(bank_network.nodes()) - failed_banks)}")

---
## 4. Sugarscape Trade Network

In the Sugarscape model, we can record every trade that occurs and build a network showing who traded with whom. Here we simulate a simple version.

In [ ]:
np.random.seed(42)
N_TRADERS = 30
N_STEPS   = 20

# Random initial endowments and metabolisms
sugar = np.random.uniform(5, 20, N_TRADERS)
spice = np.random.uniform(5, 20, N_TRADERS)
meta_s = np.random.randint(1, 4, N_TRADERS)
meta_p = np.random.randint(1, 4, N_TRADERS)

# Trade network
trade_net = nx.DiGraph()
trade_net.add_nodes_from(range(N_TRADERS))

for step in range(N_STEPS):
    # Random pairings each step
    order = np.random.permutation(N_TRADERS)
    for idx in range(0, N_TRADERS - 1, 2):
        i, j = order[idx], order[idx + 1]
        si, sj = sugar[i], sugar[j]
        pi, pj = spice[i], spice[j]
        mrs_i = (meta_s[i] / meta_p[i]) * (pi / max(si, 0.01))
        mrs_j = (meta_s[j] / meta_p[j]) * (pj / max(sj, 0.01))
        if abs(mrs_i - mrs_j) > 0.1:
            # Record the trade
            buyer = i if mrs_i > mrs_j else j
            seller = j if mrs_i > mrs_j else i
            amount = min(1.0, sugar[seller], spice[buyer])
            if trade_net.has_edge(buyer, seller):
                trade_net[buyer][seller]["volume"] += amount
                trade_net[buyer][seller]["trades"]  += 1
            else:
                trade_net.add_edge(buyer, seller, volume=amount, trades=1)
    # Consumption
    sugar -= meta_s
    spice -= meta_p
    sugar = np.maximum(sugar, 0.1)
    spice = np.maximum(spice, 0.1)

print(f"Trade network: {trade_net.number_of_nodes()} traders, {trade_net.number_of_edges()} trade links")

# Most active traders
total_trades = {n: sum(d.get("trades", 0) for _, _, d in trade_net.out_edges(n, data=True))
                + sum(d.get("trades", 0) for _, _, d in trade_net.in_edges(n, data=True))
                for n in trade_net.nodes()}

top5 = sorted(total_trades.items(), key=lambda x: -x[1])[:5]
print("\nTop 5 most active traders:")
for trader_id, trades in top5:
    print(f"  Trader {trader_id}: {trades} trades")

---
## Your Turn — Are High-Metabolism Traders More Connected?

Using the trade network and `meta_s` / `meta_p` arrays above, test whether traders with higher total metabolism (`meta_s + meta_p`) tend to have higher degree (total trade partners). Compute the Pearson correlation between `(meta_s + meta_p)` and degree centrality.

In [ ]:
# Your code here


In [ ]:
#@title Solution
from scipy import stats as sp_stats

total_metabolism = meta_s + meta_p
degree = [trade_net.degree(n) for n in range(N_TRADERS)]

r, p = sp_stats.pearsonr(total_metabolism, degree)
print(f"Correlation between total metabolism and degree: r = {r:.3f}, p = {p:.3f}")
if p < 0.05:
    direction = "positive" if r > 0 else "negative"
    print(f"Significant {direction} relationship at 5% level.")
else:
    print("No significant correlation at 5% level.")

---
## Summary

| Task | NetworkX function |
|---|---|
| Create directed graph | `nx.DiGraph()` |
| Add weighted edge | `G.add_edge(u, v, weight=w)` |
| PageRank | `nx.pagerank(G, weight='weight')` |
| Betweenness centrality | `nx.betweenness_centrality(G)` |
| Neighbours | `G.predecessors(n)`, `G.successors(n)` |
| Draw | `nx.draw_networkx_nodes/edges/labels()` |

**Reference:** Jackson, M. O. (2010). *Social and Economic Networks: Models and Analysis.* Princeton.

**Next:** NB_10_Financial_Data.ipynb — downloading and analysing market data.